# 示例 05 · Human-in-the-Loop（人工介入）

**来源：** [LangChain 官方文档 — Human-in-the-Loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)

---

## 学习目标

完成本 Notebook 后，你将能够：

1. 理解为什么 AI Agent 需要 **人工介入（Human-in-the-Loop, HITL）** 监督
2. 使用 LangGraph 原生 `interrupt()` 暂停与恢复图执行
3. 为 Agent 挂载 `HumanInTheLoopMiddleware`，实现工具调用审批
4. 处理四种决策类型：**批准（approve）**、**编辑（edit）**、**拒绝（reject）**、**直接回复（respond）**
5. 编写 **条件中断** 规则，只在高风险操作时才暂停

---

## 背景介绍

### 为什么需要人工监督？

自主 Agent 可能执行不可逆的操作——删除记录、写入文件、发送邮件。
**Human-in-the-Loop** 检查点让人类在操作真正执行之前，有机会审查、修改或取消它。

```
  Agent 规划操作
        │
        ▼
  ┌──────────────┐
  │  HITL 检查   │  ◄─── 这个工具有风险吗？
  └──────┬───────┘
         │
    ┌────┴────┐
    │   人类  │  批准 / 编辑 / 拒绝 / 直接回复
    └────┬────┘
         │
        ▼
  工具执行（或被取消）
```

### LangChain 中的两种 HITL 模式

| 模式 | 适用场景 |
|------|----------|
| **LangGraph `interrupt()`** | 自定义图结构，需要精细控制暂停位置 |
| **`HumanInTheLoopMiddleware`** | 使用 `create_agent`，需要基于策略的工具审批 |

### 四种决策类型

| 决策 | 效果 |
|------|------|
| **approve（批准）** | 按原样执行工具调用 |
| **edit（编辑）** | 修改参数后再执行 |
| **reject（拒绝）** | 取消调用并向 Agent 反馈原因 |
| **respond（直接回复）** | 用人类回答作为工具结果（"问用户" 模式） |

请从上到下依次运行每个单元格。

## 环境配置

所有导入集中在一个单元格中——重新运行可重置状态。

In [1]:
import sys
from pathlib import Path

# 将项目根目录加入 sys.path，使 common/ 包可以被导入
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# LangGraph 核心原语
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

from common.env import get_env   # 加载 .env 文件
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

handler = create_langfuse_handler()

def make_config(trace_name: str, thread_id: str = "s05-cn") -> dict:
    """构建包含 Langfuse 追踪信息的配置字典。"""
    cfg = build_langfuse_config(handler, session_id="s05-hitl-cn", trace_name=trace_name)
    cfg["configurable"] = {"thread_id": thread_id}
    return cfg

print("✓ 环境配置完成")

✓ 环境配置完成


---

## 第一部分 · LangGraph 原生 `interrupt()`

`interrupt()` 是所有 HITL 模式的基础机制。
在图节点内调用时，它会：

1. **暂停** 图在该节点的执行
2. **保存** 完整图状态到 checkpointer（检查点存储）
3. **返回** 一个 `Interrupt` 对象给调用方

图会一直保持暂停，直到你发送 `Command(resume=...)` 唤醒它。

```
graph.invoke(输入)            # 运行直到遇到 interrupt()
      │
      ▼  已暂停——状态保存在 checkpointer 中
graph.invoke(Command(resume=答案))  # 从上次停止处继续执行
```

### 步骤 1a — 构建一个简单的审批图

In [2]:
# 定义图的状态结构
class ReviewState(TypedDict):
    action: str      # Agent 想要执行的操作
    approved: bool   # 是否被批准
    result: str      # 最终结果

def plan_node(state: ReviewState) -> ReviewState:
    """Agent 规划操作（这里用固定字符串模拟）。"""
    print(f"🤖 Agent 想要执行: {state['action']}")
    return state

def approval_node(state: ReviewState) -> ReviewState:
    """在此暂停，等待人工审批。"""
    # interrupt() 暂停图并将值发送给调用方
    human_decision = interrupt({
        "question": "是否批准此操作？",
        "action": state["action"],
    })
    # 收到 Command(resume=...) 后从这里继续执行
    return {"approved": human_decision == "yes"}

def execute_node(state: ReviewState) -> ReviewState:
    """根据审批结果决定是否执行操作。"""
    if state["approved"]:
        result = f"✅ 已执行: {state['action']}"
    else:
        result = f"❌ 已取消: {state['action']}"
    print(result)
    return {"result": result}

# 构建图并指定 checkpointer（interrupt 必须依赖 checkpointer）
checkpointer = InMemorySaver()

builder = StateGraph(ReviewState)
builder.add_node("plan", plan_node)
builder.add_node("approval", approval_node)
builder.add_node("execute", execute_node)

builder.add_edge(START, "plan")
builder.add_edge("plan", "approval")
builder.add_edge("approval", "execute")
builder.add_edge("execute", END)

review_graph = builder.compile(checkpointer=checkpointer)

print("✓ 图编译完成")

✓ 图编译完成


### 步骤 1b — 运行图直到触发中断

In [3]:
# 用唯一的 thread_id 标识本次会话，状态将按 thread_id 存储
cfg = {"configurable": {"thread_id": "hitl-demo-cn-1"}}

# 第一次调用——运行 plan_node，在 approval_node 中遇到 interrupt() 后停止
result = review_graph.invoke(
    {"action": "删除 30 天前的所有订单记录", "approved": False, "result": ""},
    config=cfg,
)

# StateGraph.invoke() 返回的是普通 dict（LangGraph 1.x）
# 中断信息存储在 '__interrupt__' 键下
print("\n图已暂停。中断信息：")
print(result["__interrupt__"])

### 步骤 1c — 发送决策恢复执行

In [ ]:
# 展示中断信息，等待用户输入决策
pending = result["__interrupt__"][0].value
print(f"\n⚠️  待审批操作: {pending['action']}")
print(f"   {pending['question']}")

# input() 会阻塞，直到用户输入内容并按下 Enter
decision = input("\n你的决策（yes 批准 / no 拒绝）：").strip().lower()

# 用用户的决策恢复图——从 interrupt() 暂停处继续执行
final = review_graph.invoke(
    Command(resume=decision),
    config=cfg,
)
print("最终结果:", final["result"])

In [ ]:
# 拒绝：开启新线程并拒绝同一类操作
cfg2 = {"configurable": {"thread_id": "hitl-demo-cn-2"}}

review_graph.invoke(
    {"action": "DROP TABLE users", "approved": False, "result": ""},
    config=cfg2,
)

# 人类说不
final2 = review_graph.invoke(
    Command(resume="no"),
    config=cfg2,
)
print("最终结果:", final2["result"])

---

## 第二部分 · 使用 `HumanInTheLoopMiddleware`

`HumanInTheLoopMiddleware` 是更高层次的抽象，位于 LLM 和工具之间。
它会拦截工具调用，根据策略决定是否暂停——无需自己构建图。

```
LLM 决定调用 write_file()
        │
        ▼
  ┌──────────────────────────────┐
  │  HumanInTheLoopMiddleware    │  interrupt_on={"write_file": True}
  └──────────┬───────────────────┘
             │  触发中断
             ▼
  人类发送 Command(resume={"decisions": [{"type": "approve"}]})
             │
             ▼
       write_file() 执行
```

### 步骤 2a — 定义工具并创建 Agent

In [ ]:
# 模拟高风险工具（不产生真实副作用）
@tool
def write_file(path: str, content: str) -> str:
    """将内容写入指定路径的文件。"""
    return f"[模拟] 向 {path} 写入了 {len(content)} 字节"

@tool
def execute_sql(query: str) -> str:
    """对数据库执行 SQL 查询。"""
    return f"[模拟] 已执行 SQL: {query}"

@tool
def read_data(table: str) -> str:
    """从数据库表读取数据（只读，安全）。"""
    return f"[模拟] 从 '{table}' 读取了 42 行"

# HumanInTheLoopMiddleware 配置说明：
#   True  → 调用此工具时总是中断等待审批
#   False → 不中断，直接执行
agent = create_agent(
    model=create_llm(),
    tools=[write_file, execute_sql, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file":  True,   # 写文件前必须审批
                "execute_sql": True,   # 执行 SQL 前必须审批
                "read_data":   False,  # 只读操作，无需审批
            },
        )
    ],
    checkpointer=InMemorySaver(),   # interrupt/resume 需要 checkpointer
)

print("✓ Agent 创建完成（含 HITL 中间件）")

### 步骤 2b — 触发中断（Agent 提出高风险操作）

In [ ]:
hitl_cfg = make_config("HITL 中间件演示", thread_id="s05-cn-hitl-1")

# Agent 会决定调用 write_file，此时中间件触发中断
result = agent.invoke(
    {"messages": [{"role": "user", "content": "把汇总报告保存到 /reports/summary.txt"}]},
    config=hitl_cfg,
    version="v2",
)

print("中断信息：")
for intr in result.interrupts:
    # 打印原始中断值，方便查看其实际数据结构
    print(f"  {intr.value}")

### 步骤 2c — 决策：**批准（approve）**

使用 `{"type": "approve"}` 恢复执行，工具将按原始参数运行。

In [ ]:
# 批准工具调用
approved = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=hitl_cfg,
    version="v2",
)
# .value 是输出状态字典
approved.value["messages"][-1].pretty_print()

### 步骤 2d — 决策：**拒绝（reject）**

使用 `{"type": "reject"}` 加上 `message` 说明原因。
Agent 会收到反馈并尝试更安全的替代方案。

In [ ]:
# 新建线程演示拒绝场景
reject_cfg = make_config("HITL 拒绝演示", thread_id="s05-cn-hitl-2")

# 再次触发中间件
agent.invoke(
    {"messages": [{"role": "user", "content": "删除 orders 表中的旧记录"}]},
    config=reject_cfg,
    version="v2",
)

# 拒绝：Agent 收到反馈消息后重新规划
rejected = agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "太危险了，请改为归档而不是删除。"}]}),
    config=reject_cfg,
    version="v2",
)
rejected.value["messages"][-1].pretty_print()

### 步骤 2e — 决策：**编辑（edit）**

使用 `{"type": "edit"}` 并提供修改后的参数。
适用于工具调用方向正确但需要微调参数的情况（例如更改为更安全的路径）。

In [ ]:
edit_cfg = make_config("HITL 编辑演示", thread_id="s05-cn-hitl-3")

# Agent 提议写入 /etc/config（危险路径）
agent.invoke(
    {"messages": [{"role": "user", "content": "把配置文件写到 /etc/config.json"}]},
    config=edit_cfg,
    version="v2",
)

# 编辑：将写入路径改为更安全的工作区目录
edited = agent.invoke(
    Command(resume={"decisions": [{
        "type": "edit",
        "edited_action": {
            "name": "write_file",
            "args": {"path": "/workspace/config.json", "content": "{}"}
        }
    }]}),
    config=edit_cfg,
    version="v2",
)
edited.value["messages"][-1].pretty_print()

### 步骤 2f — 决策：**直接回复（respond）**

使用 `{"type": "respond"}` 将人类的答案作为工具的返回结果。
这实现了 **"问用户"（ask-user）** 模式——工具调用变成一轮人机对话。

In [ ]:
respond_cfg = make_config("HITL 直接回复演示", thread_id="s05-cn-hitl-4")

# 定义一个专门用于向用户提问的工具
@tool
def ask_user(question: str) -> str:
    """向用户提出澄清问题并返回其回答。"""
    return "（始终被中间件拦截）"

ask_agent = create_agent(
    model=create_llm(),
    tools=[ask_user],
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"ask_user": True})
    ],
    checkpointer=InMemorySaver(),
)

ask_agent.invoke(
    {"messages": [{"role": "user", "content": "按钮应该用什么颜色？"}]},
    config=respond_cfg,
    version="v2",
)

# 直接回复：人类的答案作为工具结果返回给 Agent
response = ask_agent.invoke(
    Command(resume={"decisions": [{"type": "respond", "message": "蓝色。"}]}),
    config=respond_cfg,
    version="v2",
)
response.value["messages"][-1].pretty_print()

---

## 第三部分 · 条件中断

每次都中断虽然安全，但会造成大量不必要的打扰。
使用 **`when`** 谓词函数，只在满足特定条件时才暂停
（例如：写入到工作区以外的路径）。

`when` 函数接收一个 `ToolCallRequest` 对象，返回 `True`（中断）或 `False`（直接执行）。

### 步骤 3a — 仅当写入路径在 `/workspace/` 之外时才中断

In [ ]:
def writes_outside_workspace(request) -> bool:
    """只有当目标路径在 /workspace/ 之外时才返回 True（触发中断）。"""
    path = request.tool_call["args"].get("path", "")
    return not path.startswith("/workspace/")

conditional_agent = create_agent(
    model=create_llm(),
    tools=[write_file, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "when": writes_outside_workspace,   # 条件谓词
                },
                "read_data": False,  # 只读操作，永不中断
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

print("✓ 条件中断 Agent 创建完成")

In [ ]:
# 情况 A：写入 /workspace/ 内——不触发中断，直接执行
safe_cfg = make_config("条件 HITL \u2014 安全路径", thread_id="s05-cn-cond-1")

safe_result = conditional_agent.invoke(
    {"messages": [{"role": "user", "content": "把日志写到 /workspace/logs/run.log"}]},
    config=safe_cfg,
    version="v2",
)
print("无中断\u2014\u2014直接执行完成：")
safe_result.value["messages"][-1].pretty_print()

In [ ]:
# 情况 B：写入 /workspace/ 之外——触发中断
risky_cfg = make_config("条件 HITL \u2014 危险路径", thread_id="s05-cn-cond-2")

risky_result = conditional_agent.invoke(
    {"messages": [{"role": "user", "content": "把凭证写入 /etc/secrets.txt"}]},
    config=risky_cfg,
    version="v2",
)
print("危险写入触发中断：")
for intr in risky_result.interrupts:
    print(f"  {intr.value}")

# 拒绝此危险写入
final = conditional_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "禁止写入 /etc 目录。"}]}),
    config=risky_cfg,
    version="v2",
)
final.value["messages"][-1].pretty_print()

---

## 总结

| 概念 | 关键 API | 适用场景 |
|------|---------|----------|
| 原生中断 | 在图节点内调用 `interrupt(value)` | 自定义图，需要精细控制暂停位置 |
| 恢复执行 | `Command(resume=答案)` | 与暂停时使用相同的 thread_id |
| 中间件审批 | `HumanInTheLoopMiddleware(interrupt_on={...})` | `create_agent` + 基于策略的审批 |
| 批准 | `{"type": "approve"}` | 按原样执行工具 |
| 拒绝 | `{"type": "reject", "message": "..."}` | 取消并向 Agent 反馈原因 |
| 编辑 | `{"type": "edit", "edited_action": {...}}` | 修改参数后执行 |
| 直接回复 | `{"type": "respond", "message": "..."}` | 人类直接回答（问用户模式） |
| 条件中断 | `"when": callable` | 仅在谓词函数返回 True 时中断 |

### 关键要求

1. **Checkpointer 必须配置** — 开发用 `InMemorySaver`，生产用 `AsyncPostgresSaver`
2. **Thread ID 必须保持一致** — 初始调用和恢复调用必须使用同一个 `thread_id`
3. **调用时指定版本** — 使用中间件时，`invoke` 或 `stream_events` 需传入 `version="v2"` 或 `version="v3"`

In [ ]:
print(f"追踪记录请访问: {get_langfuse_host()}")